# FFAI History & Analytics: Search, Cross-Analysis & Reporting

This notebook demonstrates advanced history analytics:

1. **Generate a batch of interactions** -- varied prompt names across multiple topics
2. **Search history** -- text search, prompt name filter, time-range queries
3. **Cross-prompt analysis** -- response length by prompt name, token efficiency
4. **Interaction timeline** -- timing between calls, interaction density
5. **Usage breakdown** -- input vs output tokens per prompt name
6. **Persist to Parquet** -- save all histories, reload and verify
7. **Summary report** -- a single DataFrame combining key metrics

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="coroutine.*was never awaited", category=RuntimeWarning)

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / "pyproject.toml").is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

import polars as pl  # noqa: E402

from ffai import FFAI  # noqa: E402
from ffai.Clients import FFLiteLLMClient  # noqa: E402

client = FFLiteLLMClient(model_string="mistral/mistral-small-latest")
ffai = FFAI(client=client)

print(f"Project root: {_project_root}")
print(f"Client model: {client.model}")
print("Ready")

11:16:05 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


11:16:06 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Project root: /home/aq/Documents/Source/ffai-standalone
Client model: mistral-small-latest
Ready


<div class="page-break"></div>

---

## Step 1: Generate a Batch of Interactions

Run prompts across three topic areas with different prompt names. This creates
enough data to make search and cross-analysis meaningful.

In [2]:
import time

PROMPTS = [
    ("lang_list", "Name exactly three programming languages with a one-sentence description of each."),
    ("lang_compare", "Compare Python and Rust in three sentences, focusing on performance and safety."),
    ("lang_future", "What programming paradigm do you think will dominate in 2030? Answer in two sentences."),
    ("arch_pattern", "Name two microservices patterns and when to use each. Be brief."),
    ("arch_scale", "How does horizontal scaling differ from vertical scaling? Answer in two sentences."),
    ("arch_tradeoff", "What is the biggest tradeoff when moving from monolith to microservices? One sentence."),
    ("data_type", "Explain the difference between a stack and a queue in two sentences."),
    ("data_algo", "What is the time complexity of binary search and why? Two sentences max."),
    ("data_use", "Give one real-world example where a hash table is the best data structure. One sentence."),
]

batch_results = []
start_time = time.time()

for prompt_name, prompt_text in PROMPTS:
    r = ffai.workflow.generate_response(prompt=prompt_text, prompt_name=prompt_name)
    batch_results.append(r)
    print(f"[{prompt_name:15s}] tokens={r.usage.total_tokens:4d}  cost=${r.cost_usd:.6f}  ms={r.duration_ms:.0f}")

batch_duration = time.time() - start_time
total_tokens = sum(r.usage.total_tokens for r in batch_results)
total_cost = sum(r.cost_usd for r in batch_results)
print(f"\nBatch: {len(batch_results)} prompts in {batch_duration:.1f}s")
print(f"Total: {total_tokens} tokens, ${total_cost:.6f}")

[lang_list      ] tokens= 120  cost=$0.000017  ms=917


[lang_compare   ] tokens= 255  cost=$0.000030  ms=1296


[lang_future    ] tokens= 363  cost=$0.000032  ms=1027


[arch_pattern   ] tokens= 440  cost=$0.000034  ms=899


[arch_scale     ] tokens= 513  cost=$0.000038  ms=907


[arch_tradeoff  ] tokens= 567  cost=$0.000038  ms=828


[data_type      ] tokens= 674  cost=$0.000030  ms=853


[data_algo      ] tokens= 743  cost=$0.000051  ms=744


[data_use       ] tokens= 806  cost=$0.000054  ms=772

Batch: 9 prompts in 8.3s
Total: 4481 tokens, $0.000323


<div class="page-break"></div>

---

## Step 2: Search History

`search_history()` supports filtering by text substring, prompt name, model,
and time range. All filters are optional and combine with AND logic.

In [3]:
all_history = ffai.history.history_to_dataframe()
print(f"Total interactions: {all_history.shape[0]}")

Total interactions: 9


In [4]:
print('=== Search: prompt_name == "lang_list" (exact match) ===')
lang_df = ffai.history.search_history(prompt_name="lang_list")
if lang_df.shape[0] > 0:
    print(lang_df.select([
        pl.col("prompt_name"),
        pl.col("response").str.slice(0, 80).alias("response_preview"),
    ]))
else:
    print("No results")

=== Search: prompt_name == "lang_list" (exact match) ===
shape: (1, 2)
┌─────────────┬─────────────────────────────────┐
│ prompt_name ┆ response_preview                │
│ ---         ┆ ---                             │
│ str         ┆ str                             │
╞═════════════╪═════════════════════════════════╡
│ lang_list   ┆ 1. **Python**: A high-level, i… │
└─────────────┴─────────────────────────────────┘


In [5]:
print('=== Search: text contains "scaling" in prompt or response ===')
scaling_df = ffai.history.search_history(text="scaling")
if scaling_df.shape[0] > 0:
    print(scaling_df.select([
        pl.col("prompt_name"),
        pl.col("response").str.slice(0, 80).alias("response_preview"),
    ]))
else:
    print("No results")

=== Search: text contains "scaling" in prompt or response ===
shape: (1, 2)
┌─────────────┬─────────────────────────────────┐
│ prompt_name ┆ response_preview                │
│ ---         ┆ ---                             │
│ str         ┆ str                             │
╞═════════════╪═════════════════════════════════╡
│ arch_scale  ┆ **Horizontal scaling** adds mo… │
└─────────────┴─────────────────────────────────┘


In [6]:
print('=== Search: time range (first 30 seconds) ===')
first_ts = all_history["timestamp"].min()
time_df = ffai.history.search_history(start_time=first_ts, end_time=first_ts + 30)
print(f"Found {time_df.shape[0]} interactions in first 30 seconds")
if time_df.shape[0] > 0:
    print(time_df.select([pl.col("prompt_name"), pl.col("timestamp")]))

=== Search: time range (first 30 seconds) ===
Found 9 interactions in first 30 seconds
shape: (9, 2)
┌───────────────┬───────────┐
│ prompt_name   ┆ timestamp │
│ ---           ┆ ---       │
│ str           ┆ f64       │
╞═══════════════╪═══════════╡
│ lang_list     ┆ 1.7801e9  │
│ lang_compare  ┆ 1.7801e9  │
│ lang_future   ┆ 1.7801e9  │
│ arch_pattern  ┆ 1.7801e9  │
│ arch_scale    ┆ 1.7801e9  │
│ arch_tradeoff ┆ 1.7801e9  │
│ data_type     ┆ 1.7801e9  │
│ data_algo     ┆ 1.7801e9  │
│ data_use      ┆ 1.7801e9  │
└───────────────┴───────────┘


<div class="page-break"></div>

---

## Step 3: Cross-Prompt Analysis

Analyze response characteristics across prompt names -- length distribution,
token efficiency (output tokens per character of response).

In [7]:
ordered_df = ffai.history.ordered_history_to_dataframe()

analysis = ordered_df.select([
    pl.col("prompt_name"),
    pl.col("response").str.len_chars().alias("response_chars"),
]).group_by("prompt_name").agg([
    pl.col("response_chars").mean().alias("mean_chars"),
    pl.col("response_chars").min().alias("min_chars"),
    pl.col("response_chars").max().alias("max_chars"),
    pl.col("response_chars").count().alias("count"),
]).sort("mean_chars", descending=True)

print("=== Response Length by Prompt Name ===")
print(analysis)

=== Response Length by Prompt Name ===
shape: (9, 5)
┌───────────────┬────────────┬───────────┬───────────┬───────┐
│ prompt_name   ┆ mean_chars ┆ min_chars ┆ max_chars ┆ count │
│ ---           ┆ ---        ┆ ---       ┆ ---       ┆ ---   │
│ str           ┆ f64        ┆ u32       ┆ u32       ┆ u64   │
╞═══════════════╪════════════╪═══════════╪═══════════╪═══════╡
│ lang_compare  ┆ 660.0      ┆ 660       ┆ 660       ┆ 1     │
│ lang_list     ┆ 433.0      ┆ 433       ┆ 433       ┆ 1     │
│ lang_future   ┆ 428.0      ┆ 428       ┆ 428       ┆ 1     │
│ data_type     ┆ 384.0      ┆ 384       ┆ 384       ┆ 1     │
│ arch_scale    ┆ 330.0      ┆ 330       ┆ 330       ┆ 1     │
│ arch_pattern  ┆ 316.0      ┆ 316       ┆ 316       ┆ 1     │
│ data_algo     ┆ 262.0      ┆ 262       ┆ 262       ┆ 1     │
│ arch_tradeoff ┆ 188.0      ┆ 188       ┆ 188       ┆ 1     │
│ data_use      ┆ 173.0      ┆ 173       ┆ 173       ┆ 1     │
└───────────────┴────────────┴───────────┴───────────┴───────┘


In [8]:
prompt_groups = ordered_df.group_by("prompt_name").agg([
    pl.col("prompt").str.len_chars().first().alias("prompt_chars"),
    pl.col("response").str.len_chars().first().alias("response_chars"),
]).sort("prompt_name")

with_ratio = prompt_groups.with_columns([
    (pl.col("response_chars") / pl.col("prompt_chars")).round(1).alias("expansion_ratio"),
])

print("=== Prompt-to-Response Expansion Ratio ===")
print(with_ratio)
print(f"\nAverage expansion: {with_ratio['expansion_ratio'].mean():.1f}x")

=== Prompt-to-Response Expansion Ratio ===
shape: (9, 4)
┌───────────────┬──────────────┬────────────────┬─────────────────┐
│ prompt_name   ┆ prompt_chars ┆ response_chars ┆ expansion_ratio │
│ ---           ┆ ---          ┆ ---            ┆ ---             │
│ str           ┆ u32          ┆ u32            ┆ f64             │
╞═══════════════╪══════════════╪════════════════╪═════════════════╡
│ arch_pattern  ┆ 63           ┆ 316            ┆ 5.0             │
│ arch_scale    ┆ 82           ┆ 330            ┆ 4.0             │
│ arch_tradeoff ┆ 86           ┆ 188            ┆ 2.2             │
│ data_algo     ┆ 72           ┆ 262            ┆ 3.6             │
│ data_type     ┆ 68           ┆ 384            ┆ 5.6             │
│ data_use      ┆ 88           ┆ 173            ┆ 2.0             │
│ lang_compare  ┆ 79           ┆ 660            ┆ 8.4             │
│ lang_future   ┆ 86           ┆ 428            ┆ 5.0             │
│ lang_list     ┆ 81           ┆ 433            ┆ 5.3      

<div class="page-break"></div>

---

## Step 4: Interaction Timeline

Compute timing between interactions to understand pipeline throughput.

In [9]:
timeline = ordered_df.select([
    pl.col("sequence_number"),
    pl.col("prompt_name"),
    pl.col("timestamp"),
]).sort("sequence_number")

with_delta = timeline.with_columns([
    (pl.col("timestamp") - pl.col("timestamp").shift(1)).alias("delta_seconds"),
])

print("=== Interaction Timeline ===")
print(with_delta)

deltas = with_delta.filter(pl.col("delta_seconds").is_not_null())["delta_seconds"]
if len(deltas) > 0:
    print(f"\nTime between calls: min={deltas.min():.2f}s  max={deltas.max():.2f}s  avg={deltas.mean():.2f}s")

=== Interaction Timeline ===
shape: (9, 4)
┌─────────────────┬───────────────┬───────────┬───────────────┐
│ sequence_number ┆ prompt_name   ┆ timestamp ┆ delta_seconds │
│ ---             ┆ ---           ┆ ---       ┆ ---           │
│ i64             ┆ str           ┆ f64       ┆ f64           │
╞═════════════════╪═══════════════╪═══════════╪═══════════════╡
│ 1               ┆ lang_list     ┆ 1.7801e9  ┆ null          │
│ 2               ┆ lang_compare  ┆ 1.7801e9  ┆ 1.296383      │
│ 3               ┆ lang_future   ┆ 1.7801e9  ┆ 1.027682      │
│ 4               ┆ arch_pattern  ┆ 1.7801e9  ┆ 0.900061      │
│ 5               ┆ arch_scale    ┆ 1.7801e9  ┆ 0.907771      │
│ 6               ┆ arch_tradeoff ┆ 1.7801e9  ┆ 0.829489      │
│ 7               ┆ data_type     ┆ 1.7801e9  ┆ 0.853879      │
│ 8               ┆ data_algo     ┆ 1.7801e9  ┆ 0.744246      │
│ 9               ┆ data_use      ┆ 1.7801e9  ┆ 0.773031      │
└─────────────────┴───────────────┴───────────┴──────────────

<div class="page-break"></div>

---

## Step 5: Usage Breakdown

Build a per-prompt-name usage table from the raw history, which includes
the `usage` TokenUsage object per interaction.

In [10]:
usage_rows = []
for (prompt_name, _), r in zip(PROMPTS, batch_results):
    usage_rows.append({
        "prompt_name": prompt_name,
        "input_tokens": r.usage.input_tokens,
        "output_tokens": r.usage.output_tokens,
        "total_tokens": r.usage.total_tokens,
        "cost_usd": r.cost_usd,
        "duration_ms": r.duration_ms,
    })

usage_df = pl.DataFrame(usage_rows)
print(usage_df)

shape: (9, 6)
┌───────────────┬──────────────┬───────────────┬──────────────┬───────────┬─────────────┐
│ prompt_name   ┆ input_tokens ┆ output_tokens ┆ total_tokens ┆ cost_usd  ┆ duration_ms │
│ ---           ┆ ---          ┆ ---           ┆ ---          ┆ ---       ┆ ---         │
│ str           ┆ i64          ┆ i64           ┆ i64          ┆ f64       ┆ f64         │
╞═══════════════╪══════════════╪═══════════════╪══════════════╪═══════════╪═════════════╡
│ lang_list     ┆ 38           ┆ 82            ┆ 120          ┆ 0.000017  ┆ 916.7       │
│ lang_compare  ┆ 136          ┆ 119           ┆ 255          ┆ 0.00003   ┆ 1295.9      │
│ lang_future   ┆ 277          ┆ 86            ┆ 363          ┆ 0.0000321 ┆ 1027.2      │
│ arch_pattern  ┆ 379          ┆ 61            ┆ 440          ┆ 0.000034  ┆ 899.1       │
│ arch_scale    ┆ 456          ┆ 57            ┆ 513          ┆ 0.000038  ┆ 907.4       │
│ arch_tradeoff ┆ 533          ┆ 34            ┆ 567          ┆ 0.0000381 ┆ 828.3     

In [11]:
usage_by_prompt = usage_df.group_by("prompt_name").agg([
    pl.col("input_tokens").sum().alias("total_input"),
    pl.col("output_tokens").sum().alias("total_output"),
    pl.col("total_tokens").sum().alias("total_tokens"),
    pl.col("cost_usd").sum().round(6).alias("total_cost"),
    pl.col("duration_ms").sum().round(0).alias("total_ms"),
]).sort("total_tokens", descending=True)

with_efficiency = usage_by_prompt.with_columns([
    (pl.col("total_output") / pl.col("total_input")).round(2).alias("out_in_ratio"),
])

print("=== Usage by Prompt Name ===")
print(with_efficiency)

=== Usage by Prompt Name ===
shape: (9, 7)
┌───────────────┬─────────────┬──────────────┬──────────────┬────────────┬──────────┬──────────────┐
│ prompt_name   ┆ total_input ┆ total_output ┆ total_tokens ┆ total_cost ┆ total_ms ┆ out_in_ratio │
│ ---           ┆ ---         ┆ ---          ┆ ---          ┆ ---        ┆ ---      ┆ ---          │
│ str           ┆ i64         ┆ i64          ┆ i64          ┆ f64        ┆ f64      ┆ f64          │
╞═══════════════╪═════════════╪══════════════╪══════════════╪════════════╪══════════╪══════════════╡
│ data_use      ┆ 763         ┆ 43           ┆ 806          ┆ 0.000054   ┆ 772.0    ┆ 0.06         │
│ data_algo     ┆ 691         ┆ 52           ┆ 743          ┆ 0.000051   ┆ 744.0    ┆ 0.08         │
│ data_type     ┆ 583         ┆ 91           ┆ 674          ┆ 0.00003    ┆ 853.0    ┆ 0.16         │
│ arch_tradeoff ┆ 533         ┆ 34           ┆ 567          ┆ 0.000038   ┆ 828.0    ┆ 0.06         │
│ arch_scale    ┆ 456         ┆ 57           ┆ 5

<div class="page-break"></div>

---

## Step 6: Persist to Parquet

Write all four history stores to Parquet files, then reload and verify.

In [12]:
import tempfile

persist_dir = tempfile.mkdtemp(prefix="ffai_history_")

ffai._exporter.persist_dir = persist_dir
ffai._exporter.persist_name = "analytics"

success = ffai.history.persist_all_histories()
print(f"Persist succeeded: {success}")
print(f"Persist dir: {persist_dir}")

Persist succeeded: True
Persist dir: /tmp/ffai_history_sfqbau5b


In [13]:
from pathlib import Path as P

persist_path = P(persist_dir)
parquet_files = sorted(persist_path.glob("*.parquet"))
print(f"Parquet files ({len(parquet_files)}):")
for f in parquet_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size_kb:.1f} KB")

Parquet files (4):
  analytics_clean_history.parquet          8.1 KB
  analytics_history.parquet                8.1 KB
  analytics_ordered.parquet                6.0 KB
  analytics_prompt_attr.parquet            5.6 KB


In [14]:
reloaded = pl.read_parquet(persist_path / "analytics_ordered.parquet")
print(f"Reloaded ordered history: {reloaded.shape[0]} rows x {reloaded.shape[1]} columns")
print(f"Columns: {list(reloaded.columns)}")
print()
display_cols = [c for c in ["sequence_number", "prompt_name", "model", "datetime"] if c in reloaded.columns]
print(reloaded.select(display_cols))

Reloaded ordered history: 9 rows x 8 columns
Columns: ['sequence_number', 'model', 'timestamp', 'prompt_name', 'prompt', 'response', 'history', 'datetime']

shape: (9, 4)
┌─────────────────┬───────────────┬──────────────────────┬────────────────────────────┐
│ sequence_number ┆ prompt_name   ┆ model                ┆ datetime                   │
│ ---             ┆ ---           ┆ ---                  ┆ ---                        │
│ i64             ┆ str           ┆ str                  ┆ datetime[μs]               │
╞═════════════════╪═══════════════╪══════════════════════╪════════════════════════════╡
│ 1               ┆ lang_list     ┆ mistral-small-latest ┆ 2026-05-29 18:16:10.130910 │
│ 2               ┆ lang_compare  ┆ mistral-small-latest ┆ 2026-05-29 18:16:11.427294 │
│ 3               ┆ lang_future   ┆ mistral-small-latest ┆ 2026-05-29 18:16:12.454975 │
│ 4               ┆ arch_pattern  ┆ mistral-small-latest ┆ 2026-05-29 18:16:13.355036 │
│ 5               ┆ arch_scale    ┆ m

In [15]:
import shutil

shutil.rmtree(persist_dir)
print(f"Cleaned up {persist_dir}")

Cleaned up /tmp/ffai_history_sfqbau5b


<div class="page-break"></div>

---

## Step 7: Summary Report

Combine key metrics into a single report DataFrame.

In [16]:
summary_rows = []
for (prompt_name, _), r in zip(PROMPTS, batch_results):
    summary_rows.append({
        "prompt_name": prompt_name,
        "input_tokens": r.usage.input_tokens,
        "output_tokens": r.usage.output_tokens,
        "total_tokens": r.usage.total_tokens,
        "cost_usd": round(r.cost_usd, 6),
        "duration_ms": round(r.duration_ms, 1),
        "response_chars": len(r.response),
    })

report_df = pl.DataFrame(summary_rows)

report_with_metrics = report_df.with_columns([
    (pl.col("output_tokens") / pl.col("input_tokens")).round(2).alias("out_in_ratio"),
    (pl.col("response_chars") / pl.col("output_tokens")).round(1).alias("chars_per_output_token"),
])

totals_row = {
    "prompt_name": "TOTAL",
    "input_tokens": report_df["input_tokens"].sum(),
    "output_tokens": report_df["output_tokens"].sum(),
    "total_tokens": report_df["total_tokens"].sum(),
    "cost_usd": round(report_df["cost_usd"].sum(), 6),
    "duration_ms": round(report_df["duration_ms"].sum(), 1),
    "response_chars": report_df["response_chars"].sum(),
    "out_in_ratio": round(report_df["output_tokens"].sum() / report_df["input_tokens"].sum(), 2),
    "chars_per_output_token": round(report_df["response_chars"].sum() / report_df["output_tokens"].sum(), 1),
}

final_report = pl.concat([report_with_metrics, pl.DataFrame([totals_row])], how="diagonal")
print("=== Pipeline Summary Report ===")
print(final_report)

=== Pipeline Summary Report ===
shape: (10, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ prompt_na ┆ input_tok ┆ output_to ┆ total_tok ┆ … ┆ duration_ ┆ response_ ┆ out_in_ra ┆ chars_pe │
│ me        ┆ ens       ┆ kens      ┆ ens       ┆   ┆ ms        ┆ chars     ┆ tio       ┆ r_output │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ _token   │
│ str       ┆ i64       ┆ i64       ┆ i64       ┆   ┆ f64       ┆ i64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ lang_list ┆ 38        ┆ 82        ┆ 120       ┆ … ┆ 916.7     ┆ 433       ┆ 2.16      ┆ 5.3      │
│ lang_comp ┆ 136       ┆ 119       ┆ 255       ┆ … ┆ 1295.9    ┆ 660       ┆ 0.88      ┆ 5.5      │
│ are       ┆           ┆           ┆       

<div class="page-break"></div>

---

## Key Takeaways

| Capability | Method / Pattern | Use Case |
|---|---|---|
| Text search | `search_history(text="keyword")` | Find interactions mentioning a topic |
| Name filter | `search_history(prompt_name="x")` | Retrieve all calls for a specific prompt |
| Time range | `search_history(start_time=ts, end_time=ts)` | Analyze a specific time window |
| Cross-prompt stats | `ordered_history_to_dataframe()` + Polars `group_by` | Compare response characteristics across prompts |
| Expansion ratio | `response_chars / prompt_chars` | Measure how verbose the model is relative to input |
| Timeline deltas | `timestamp` differences between sequences | Understand pipeline throughput |
| Output efficiency | `output_tokens / input_tokens` | Compare token economy across prompt names |
| Parquet persistence | `persist_all_histories()` | Archive history for offline analysis or audit |
| Summary report | Combine per-turn metrics into one DataFrame | One-view pipeline health check |